
# Predicting Child Stunting Growth in Kenya

## Problem Statement

Predicting stunting severity (severe stunting, moderate stunting, or normal growth, per WHO height-for-age z-score thresholds of -3 SD and -2 SD) for Kenyan children under five, using household, maternal, and child-health variables from the Kenya Demographic and Health Survey Children's Recode — including region, residence type, mother's education, household wealth index, drinking water source, toilet facility, birth order and spacing, birth weight, breastfeeding duration, and antenatal care history. Primary metric: macro-averaged F1 score, reported alongside one-vs-rest ROC-AUC and per-class precision/recall.

## The Problem

Stunting — being too short for one's age because of chronic undernutrition —
still affects roughly one in five children under five in Kenya. A stunted
child faces lifelong risks: weaker immunity, slower cognitive development,
and lower earning potential as an adult. Health workers and parents need a
reliable way to spot which children are stunted, and how severely, early
enough to act — but with limited time and resources, they cannot manually
review every household's full survey record to catch every at-risk child.

## Our Solution

We build a machine learning model that takes a child's household, maternal,
and health-history information and classifies their growth status as
**severe stunting**, **moderate stunting**, or **normal growth**. Beyond the
prediction itself, each outcome is paired with practical guidance: children
flagged as stunted (severe or moderate) receive diet and care recommendations
aimed at correcting growth faltering, while children predicted to be growing
normally receive guidance to continue their current feeding practices. The
goal is a tool that medical practitioners and parents can use to decide, at a
glance, whether a child's diet needs to change and how.

## Data Source

The data comes from the **Kenya Demographic and Health Survey (KDHS)**,
specifically its **Children's Recode** file — a nationally representative
household survey that records one row per live birth reported by an
interviewed woman (`bidx` = birth order index within her birth history,
`caseid` identifies the mother). For children under five it captures health,
nutrition, immunization, and anthropometric (height/weight) measurements
alongside household and maternal background variables. The Children's Recode
was originally distributed as a Stata file (`KEKR8CFL.DTA`, with column
layout described in `KEKR8CFL.DCT`); we use those two files only once, up
front, to decode Stata's numeric variable and value codes into their
plain-English meanings — from that point on we work entirely from CSV.

**Inputs used**

| File | Role |
|---|---|
| `child_recode_raw.csv` | the raw data we are cleaning (19,530 rows × 1,312 columns) |
| `variable_labels.csv` | plain-English description for every variable code (`v106` -> "Highest educational level"), used to understand and rename the columns we keep |



In [5]:

# --- Imports -----------------------------------------------------------
!pip install pyreadstat
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadstat  # only used to read METADATA (variable/value labels) from the .dta file
from sklearn.exceptions import UndefinedMetricWarning

# In a few cross-validation folds, "severe stunting" (the rarest class) can
# end up with zero predicted samples for a weaker model; sklearn's macro
# precision/recall then emits a warning and reports 0.0 for that class,
# which is the mathematically correct (if uninformative) value - suppressed
# here so it doesn't clutter otherwise-successful cell output.
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 17.7 MB/s eta 0:00:00



# Part 1: Data Cleaning and Preprocessing

This part takes the raw export and turns it into a trustworthy, analysis-ready
table: fixing identifiers, removing dead columns, converting DHS placeholder
codes to real `NaN`s, right-sizing dtypes, and narrowing down to the variables
the stunting model actually needs.



## 1. Load the raw data and the variable-label dictionary


In [6]:
#Load Data
# --- File locations ------------------------------------------------------
# Adjust these if your files live somewhere else. All four inputs are expected
# to sit in the same folder as this notebook by default.
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/Ngao_labs/capstone/child recode/")
RAW_CSV        = DATA_DIR / "child_recode_raw.csv"
LABELS_CSV     = DATA_DIR / "variable_labels.csv"
DTA_FILE       = DATA_DIR / "KEKR8CFL.DTA"

OUT_DIR = Path("/content/drive/MyDrive/Ngao_labs/capstone/child recode/processed")
OUT_DIR.mkdir(exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:

# The raw export only has two genuinely text columns (caseid, v000); every
# other one of the 1,312 columns is numeric (DHS encodes everything -
# including categories - as integer codes). We let pandas infer dtypes on
# the first pass and fix them up deliberately in later steps.
df = pd.read_csv(RAW_CSV, low_memory=False)

# variable -> plain-English description, e.g. "v106" -> "Highest educational level"
var_labels = pd.read_csv(LABELS_CSV).set_index("variable")["label"].to_dict()

print(f"Raw data: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Variable descriptions loaded for {len(var_labels):,} variables")
df.head(3)


Raw data: 19,530 rows x 1,312 columns
Variable descriptions loaded for 1,312 variables


,caseid,bidx,v000,v001,v002,v003,v004,v005,v006,v007,v008,v008a,v009,v010,v011,v012,v013,v014,v015,v016,v017,v018,v019,v019a,v020,v021,v022,v023,v024,v025,...,s446,s454a,idx95,s509y,sd509y,sm509y,sy509y,s528a,s509d,sd509d,sm509d,sy509d,s607aa,s607ba,s612q,s612r,s612s,s612t,s615e,s621a,s621b,s626q,s626r,s626s,s626t,s631a,s631b,s631c,s631l,s631m
0,1 4 2,1,KE8,1,4,2,1,1296049,4,2022,1468,44676,10,1987,1054,34,4,3,1,25,1405,17,64,2,0,1,1,1,1,1,...,2.0,203.0,1.0,0.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 13 2,1,KE8,1,13,2,1,1296049,4,2022,1468,44679,12,1982,996,39,5,1,1,28,1405,17,64,2,0,1,1,1,1,1,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 26 2,1,KE8,1,26,2,1,1296049,4,2022,1468,44676,11,1993,1127,28,3,1,1,25,1405,17,64,2,0,1,1,1,1,1,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



## 2. Initial audit

Before changing anything, get a baseline: are there duplicate records, how
much is already missing, and how big is the file in memory.


In [9]:

# --- Duplicate check ------------------------------------------------------
# Each row should be a unique (mother, birth) pair.
n_dupes = df.duplicated(subset=["caseid", "bidx"]).sum()
print(f"Duplicate (caseid, bidx) pairs: {n_dupes}")

# --- Baseline missingness & memory ----------------------------------------
baseline_missing_cells = df.isnull().sum().sum()
baseline_missing_pct = 100 * baseline_missing_cells / df.size
baseline_memory_mb = df.memory_usage(deep=True).sum() / 1e6

print(f"Missing cells (raw, before recoding placeholder codes): "
      f"{baseline_missing_cells:,} ({baseline_missing_pct:.1f}% of all cells)")
print(f"Memory usage: {baseline_memory_mb:.1f} MB")


Duplicate (caseid, bidx) pairs: 0
Missing cells (raw, before recoding placeholder codes): 16,083,722 (62.8% of all cells)
Memory usage: 206.9 MB



## 3. Build a data dictionary from the Stata metadata

The `.dta` file's *metadata* (not its data rows) tells us, for every
categorical variable, exactly what each numeric code means — including
which codes are placeholders like "don't know" or "flagged case" rather
than real answers. We read this once with `pyreadstat` using
`metadataonly=True`, which is essentially instant since it never touches
the 28 MB of actual row data.


In [10]:

_, meta = pyreadstat.read_dta(str(DTA_FILE), metadataonly=True)

# variable -> {code: "meaning"}, e.g. "b4" -> {1: "male", 2: "female"}
value_labels = meta.variable_value_labels

print(f"Variables with a value-label set: {len(value_labels):,} / {len(meta.column_names):,}")

# Peek at a couple of examples to see the kind of thing we're working with
for v in ["b4", "hw70", "m19"]:
    print(f"  {v}: {value_labels.get(v)}")


Variables with a value-label set: 1,206 / 1,312
  b4: {1: 'male', 2: 'female'}
  hw70: {9996: 'height out of plausible limits', 9997: 'age in days out of plausible limits', 9998: 'flagged cases'}
  m19: {9996: 'not weighed at birth', 9998: "don't know"}


In [11]:

# Persist a full data dictionary (description + value labels where they exist)
# so anyone using the cleaned CSV later can look up what a column/code means
# without needing the original .dta file.
data_dictionary = {}
for col in df.columns:
    data_dictionary[col] = {
        "label": var_labels.get(col, ""),
        "value_labels": value_labels.get(col, {}),
    }

with open(OUT_DIR / "data_dictionary.json", "w") as f:
    json.dump(data_dictionary, f, indent=2)

print(f"Data dictionary written for {len(data_dictionary):,} columns -> "
      f"{OUT_DIR / 'data_dictionary.json'}")


Data dictionary written for 1,312 columns -> /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/data_dictionary.json



## 4. Clean identifier columns

`caseid` is a fixed-width string (it packs cluster/household/line-number
together) and can pick up leading/trailing spaces when exported to CSV.
`v000` (country + survey phase code) is the other text column. Both just
need whitespace stripped. We also build a single tidy `child_id` key.


In [12]:

df["caseid"] = df["caseid"].astype(str).str.strip()
df["v000"]   = df["v000"].astype(str).str.strip()

# A convenient, unambiguous per-child key: mother's caseid + this birth's index
child_id = (df["caseid"] + "_" + df["bidx"].astype("Int64").astype(str)).rename("child_id")
df = pd.concat([child_id, df], axis=1)

assert df["child_id"].is_unique, "child_id should uniquely identify each row"
print("child_id created and confirmed unique.")
df[["child_id", "caseid", "bidx"]].head(3)


child_id created and confirmed unique.


,child_id,caseid,bidx
0,1 4 2_1,1 4 2,1
1,1 13 2_1,1 13 2,1
2,1 26 2_1,1 26 2,1



## 5. Drop columns that carry no information

Two kinds of "dead weight" columns show up in DHS exports:

- **Fully empty columns** — usually variables not fielded in this particular
  survey (e.g. modules asked in some country-phases but not others).
- **Constant columns** — every non-missing row has the exact same value
  (e.g. a survey-phase code, always `"KE8"`). They carry zero variance, so
  they add size without adding analytical value.

We drop both, but log exactly what we dropped (and, for constants, what the
one value was) so nothing disappears silently.


In [13]:

protected_cols = {"child_id", "caseid", "bidx"}  # never drop identifiers

fully_null_cols = [c for c in df.columns if c not in protected_cols and df[c].isnull().all()]

constant_cols = [
    c for c in df.columns
    if c not in protected_cols and c not in fully_null_cols and df[c].nunique(dropna=True) <= 1
]

dropped_log = []
for c in fully_null_cols:
    dropped_log.append({"column": c, "reason": "100% empty", "label": var_labels.get(c, "")})
for c in constant_cols:
    only_value = df[c].dropna().iloc[0] if df[c].notna().any() else None
    dropped_log.append({
        "column": c, "reason": f"constant (always {only_value!r})", "label": var_labels.get(c, "")
    })

dropped_df = pd.DataFrame(dropped_log)

cols_to_drop = fully_null_cols + constant_cols
df = df.drop(columns=cols_to_drop)

print(f"Dropped {len(fully_null_cols)} fully-empty columns and "
      f"{len(constant_cols)} constant columns ({len(cols_to_drop)} total).")
print(f"Remaining columns: {df.shape[1]:,}")
dropped_df.to_csv(OUT_DIR / "dropped_columns_log.csv", index=False)
dropped_df.head(10)


Dropped 354 fully-empty columns and 51 constant columns (405 total).
Remaining columns: 908


,column,reason,label
0,v026,100% empty,NA - De facto place of residence
1,v029,100% empty,NA - Keyer identification
2,v031,100% empty,NA - Field editor
3,v032,100% empty,NA - Office editor
4,v048,100% empty,NA - Team supervisor
5,v103,100% empty,NA - Childhood place of residence
6,v134,100% empty,NA - De facto place of residence
7,v156,100% empty,NA - Ever participated in a literacy program o...
8,v167,100% empty,NA - Number of trips in last 12 months
9,v168,100% empty,NA - Away for more than one month in last 12 m...



## 6. Recode DHS placeholder codes to proper `NaN`

DHS variables use reserved numeric codes for non-answers instead of a blank
cell — for example `hw70` (height-for-age z-score) uses `9996` = "height out
of plausible limits", `9997` = "age in days out of plausible limits", and
`9998` = "flagged cases". Left as numbers, these would silently corrupt any
average, histogram, or model fit on that column.

Rather than guessing at "big numbers look like missing codes", we use the
actual value-label text pulled from the `.dta` metadata in step 3 and match
it against a set of known missingness phrases. This only converts codes
that are *documented* as non-substantive — real category codes (like
`v106` = 0 "no education") are left alone.


In [14]:

# Phrases that reliably indicate "this code is not a real answer" in DHS codebooks.
MISSING_PATTERN = re.compile(
    r"\bmissing\b|not applicable|not present|\brefused\b|\bflagged\b|"
    r"plausible limits|don.?t know|not weighed|not measured|\binconsistent\b|"
    r"no response|not stated|not dejure",
    re.IGNORECASE,
)

# Build {column: {code: label}} for just the codes that match the pattern above
missing_code_map = {}
for col, labels in value_labels.items():
    if col not in df.columns:
        continue
    flagged = {code: text for code, text in labels.items() if MISSING_PATTERN.search(str(text))}
    if flagged:
        missing_code_map[col] = flagged

print(f"{len(missing_code_map):,} columns contain at least one documented missing/placeholder code.")


401 columns contain at least one documented missing/placeholder code.


In [15]:

# Apply the recoding, keeping a full audit trail of exactly how many cells
# changed per column and per code.
recode_log = []
for col, flagged in missing_code_map.items():
    codes = list(flagged.keys())
    hit_counts = df[col].isin(codes)
    n_hits = int(hit_counts.sum())
    if n_hits == 0:
        continue
    for code, text in flagged.items():
        n = int((df[col] == code).sum())
        if n > 0:
            recode_log.append({
                "column": col, "code": code, "meaning": text, "n_cells_recoded": n,
            })
    df.loc[hit_counts, col] = np.nan

recode_log_df = pd.DataFrame(recode_log).sort_values("n_cells_recoded", ascending=False)
recode_log_df.to_csv(OUT_DIR / "missing_code_recode_log.csv", index=False)

print(f"Recoded {recode_log_df['n_cells_recoded'].sum():,} cells across "
      f"{recode_log_df['column'].nunique():,} columns to NaN.")
recode_log_df.head(15)


Recoded 22,094 cells across 332 columns to NaN.


,column,code,meaning,n_cells_recoded
67,v621,8,don't know,1389
90,m19,9996,not weighed at birth,1266
92,m19a,0,not weighed,1266
15,v379,98,don't know,1199
16,v380,8,don't know,1199
8,v244,8,don't know,654
6,v217,8,don't know,649
59,v529,997,inconsistent,600
2,v139,97,not dejure resident,538
7,v226,997,inconsistent,535



## 7. Confirm the effect: missingness before vs. after

A quick before/after comparison shows the recoding step actually surfaced
missingness that was previously hiding behind numeric placeholder codes.


In [16]:

after_missing_cells = df.isnull().sum().sum()
after_missing_pct = 100 * after_missing_cells / df.size

comparison = pd.DataFrame({
    "stage": ["raw (blank cells only)", "after placeholder-code recoding"],
    "missing_cells": [baseline_missing_cells, after_missing_cells],
    "pct_of_all_cells": [round(baseline_missing_pct, 2), round(after_missing_pct, 2)],
})
comparison


,stage,missing_cells,pct_of_all_cells
0,raw (blank cells only),16083722,62.77
1,after placeholder-code recoding,8765073,49.43



## 8. Right-size data types

Now that missing values are proper `NaN`s rather than large placeholder
integers, we can shrink numeric columns to the smallest safe type. Columns
with any `NaN` need pandas's **nullable** integer type (`Int64`, capital I)
rather than plain `int64`, since plain numpy ints can't hold `NaN`.


In [17]:

before_mb = df.memory_usage(deep=True).sum() / 1e6

for col in df.columns:
    if col in ("child_id", "caseid", "v000"):
        continue  # leave text columns alone
    series = df[col]
    if not pd.api.types.is_numeric_dtype(series):
        continue

    if (series.dropna() % 1 == 0).all():
        # Whole numbers -> nullable integer, sized to the observed range
        max_abs = series.abs().max()
        if pd.isna(max_abs):
            continue
        if max_abs < 2**7:
            dtype = "Int8"
        elif max_abs < 2**15:
            dtype = "Int16"
        elif max_abs < 2**31:
            dtype = "Int32"
        else:
            dtype = "Int64"
        df[col] = series.astype(dtype)
    else:
        # Genuine decimals (e.g. sample weights) -> float32 is ample precision here
        df[col] = series.astype("float32")

after_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"Memory usage: {before_mb:.1f} MB -> {after_mb:.1f} MB "
      f"({100 * (1 - after_mb / before_mb):.0f}% smaller)")


Memory usage: 143.9 MB -> 40.0 MB (72% smaller)



## 9. Add human-readable labels for the most commonly used variables

Keeping the underlying columns numeric-coded matches how the DHS data is
normally shared and merged with companion files, but a handful of variables
get referenced constantly in child-health analysis. For just those, we add
a companion `<var>_label` column decoded straight from the value-label
dictionary, so they're readable without a lookup step.


In [18]:

KEY_VARS_TO_DECODE = [
    "b4",     # sex of child
    "b5",     # child is alive
    "v106",   # mother's highest educational level
    "v190",   # wealth index quintile (combined)
    "v190a",  # wealth index quintile (urban/rural specific)
    "v024",   # region
    "v025",   # type of place of residence
    "v013",   # mother's age in 5-year groups
    "v113",   # source of drinking water
    "v116",   # type of toilet facility
    "h11",    # had diarrhea recently
    "h22",    # had fever in last two weeks
]

decoded_count = 0
for var in KEY_VARS_TO_DECODE:
    if var not in df.columns or var not in value_labels:
        continue
    label_map = value_labels[var]
    df[f"{var}_label"] = df[var].map(label_map)
    decoded_count += 1

print(f"Added readable *_label columns for {decoded_count} key variables.")
df[[v for pair in ((v, f"{v}_label") for v in KEY_VARS_TO_DECODE if v in df.columns)
      for v in pair]].head(5)


Added readable *_label columns for 12 key variables.


,b4,b4_label,b5,b5_label,v106,v106_label,v190,v190_label,v190a,v190a_label,v024,v024_label,v025,v025_label,v013,v013_label,v113,v113_label,v116,v116_label,h11,h11_label,h22,h22_label
0,1,male,1,yes,0,no education,4,richer,2,poorer,1,mombasa,1,urban,4,30-34,14,public tap/standpipe,22,pit latrine with slab,0,no,0,no
1,2,female,1,yes,2,secondary,5,richest,4,richer,1,mombasa,1,urban,5,35-39,12,piped to yard/plot,12,flush to septic tank,0,no,0,no
2,1,male,1,yes,2,secondary,4,richer,2,poorer,1,mombasa,1,urban,3,25-29,13,piped to neighbor,21,ventilated improved pit latrine (vip),0,no,0,no
3,2,female,1,yes,2,secondary,5,richest,4,richer,1,mombasa,1,urban,4,30-34,14,public tap/standpipe,21,ventilated improved pit latrine (vip),0,no,1,yes
4,1,male,1,yes,2,secondary,5,richest,3,middle,1,mombasa,1,urban,4,30-34,14,public tap/standpipe,21,ventilated improved pit latrine (vip),0,no,0,no



## 10. Sanity-check a few key variables

A quick range/consistency check on variables that matter most for child
health analysis. This isn't exhaustive (that would mean hand-checking all
~1,300 variables), but it catches the kind of gross problem worth knowing
about before analysis: e.g. z-scores outside their documented plausible
range, or a sex code that isn't 1/2.


In [19]:

checks = []

# Anthropometric z-scores are stored *100 and should sit roughly in [-600, 600]
# once flagged/implausible codes have already been removed in step 6.
for zcol, zname in [("hw70", "height-for-age"), ("hw71", "weight-for-age"), ("hw72", "weight-for-height")]:
    if zcol in df.columns:
        out_of_range = df[zcol].dropna()
        bad = ((out_of_range < -600) | (out_of_range > 600)).sum()
        checks.append({"check": f"{zname} z-score ({zcol}) within [-6.00, 6.00] SD", "violations": int(bad)})

# Sex of child should only ever be 1 (male) or 2 (female)
if "b4" in df.columns:
    bad = (~df["b4"].isin([1, 2]) & df["b4"].notna()).sum()
    checks.append({"check": "b4 (sex of child) is 1 or 2", "violations": int(bad)})

# Child's current age in years should be non-negative and plausible (<= ~20 for a KR file)
if "b8" in df.columns:
    bad = ((df["b8"] < 0) | (df["b8"] > 20)).sum()
    checks.append({"check": "b8 (child's current age in years) within [0, 20]", "violations": int(bad)})

checks_df = pd.DataFrame(checks)
checks_df


,check,violations
0,"height-for-age z-score (hw70) within [-6.00, 6...",0
1,"weight-for-age z-score (hw71) within [-6.00, 6...",0
2,weight-for-height z-score (hw72) within [-6.00...,0
3,b4 (sex of child) is 1 or 2,0
4,"b8 (child's current age in years) within [0, 20]",0



## 11. Keep only the features listed in `features.docx`

The project's feature list (`features.docx`) specifies exactly one target
variable and a curated set of predictors for a child-stunting model. We
narrow the cleaned dataset down to just those columns before saving.

- **Target:** `hw70` (height-for-age z-score — the stunting outcome).
- **Predictors:** the child-, maternal-, household-, and illness-level
  variables listed in the document.
- `v455` is mentioned in the document itself as "consider if present in
  dataset" — it isn't one of this survey's columns, so it's skipped.
- We keep `child_id`, `caseid`, and `bidx` even though they aren't in the
  feature document, since they're the only way to identify each row —
  a feature table with no key isn't usable. We also keep the `_label`
  companion columns (e.g. `b4_label`) added in step 9 for any requested
  variable that has one, since those are just a readable version of a
  variable already on the list, not a new one.


In [20]:

# Exactly the variables named in features.docx
TARGET_VAR = "hw70"

FEATURE_VARS = [
    # Child-level
    "hw1", "b4", "bord", "b0", "b11", "m18", "m19", "m19a",
    # Maternal / care-seeking
    "v106", "v133", "v437", "v438", "m14", "m13", "m45", "m4", "m5",
    # Household / socioeconomic
    "v190", "v113", "v116", "v025", "v024", "v151",
    # Health / illness
    "h11", "h33", "h10",
]

# v455 is called out in the document as "consider if present in dataset" —
# it isn't one of this survey's variables, so there's nothing to keep.
requested = [TARGET_VAR] + FEATURE_VARS
missing_from_data = [v for v in requested if v not in df.columns]
if missing_from_data:
    print(f"Note: not present in this dataset, skipped: {missing_from_data}")

# Identifiers we keep regardless (not "features", but required to use the table at all)
id_cols = [c for c in ["child_id", "caseid", "bidx"] if c in df.columns]

# Readable *_label companions for any requested variable that has one
label_cols = [f"{v}_label" for v in requested if f"{v}_label" in df.columns]

keep_cols = id_cols + [v for v in requested if v in df.columns] + label_cols
dropped_for_features = [c for c in df.columns if c not in keep_cols]

df = df[keep_cols]

print(f"Kept {len(keep_cols)} columns ({len(id_cols)} identifiers, "
      f"{len(requested) - len(missing_from_data)} requested features, {len(label_cols)} label companions).")
print(f"Dropped {len(dropped_for_features)} columns not on the features list.")
df.head(3)


Kept 38 columns (3 identifiers, 27 requested features, 8 label companions).
Dropped 882 columns not on the features list.


,child_id,caseid,bidx,hw70,hw1,b4,bord,b0,b11,m18,m19,m19a,v106,v133,v437,v438,m14,m13,m45,m4,m5,v190,v113,v116,v025,v024,v151,h11,h33,h10,b4_label,v106_label,v190_label,v113_label,v116_label,v025_label,v024_label,h11_label
0,1 4 2_1,1 4 2,1,362,8,1,4,0,92,2,4000,2,0,0,973,1614,5,5,1,95,7,4,14,22,1,1,1,0,0,<NA>,male,no education,richer,public tap/standpipe,pit latrine with slab,urban,mombasa,no
1,1 13 2_1,1 13 2,1,-11,36,2,5,0,30,<NA>,<NA>,<NA>,2,12,1028,1613,<NA>,<NA>,<NA>,<NA>,<NA>,5,12,12,1,1,1,0,<NA>,<NA>,female,secondary,richest,piped to yard/plot,flush to septic tank,urban,mombasa,no
2,1 26 2_1,1 26 2,1,-86,58,1,3,0,23,<NA>,<NA>,<NA>,2,12,660,1492,<NA>,<NA>,<NA>,<NA>,<NA>,4,13,21,1,1,1,0,<NA>,<NA>,male,secondary,richer,piped to neighbor,ventilated improved pit latrine (vip),urban,mombasa,no



## 12. Save the cleaned dataset and supporting files

Everything goes into `cleaned_output/`:

| File | Contents |
|---|---|
| `child_recode_clean.csv` | the cleaned dataset, narrowed to the `features.docx` variable list plus identifiers |
| `data_dictionary.json` | variable label + value-label lookup (kept for every column seen during cleaning, so it still documents dropped columns too) |
| `dropped_columns_log.csv` | every column removed in step 5, and why |
| `missing_code_recode_log.csv` | every placeholder code converted to `NaN` in step 6, and how many cells it affected |


In [21]:

out_csv = OUT_DIR / "child_recode_clean.csv"
df.to_csv(out_csv, index=False)

print("Saved:")
print(f"  {out_csv}  ({df.shape[0]:,} rows x {df.shape[1]:,} columns)")
print(f"  {OUT_DIR / 'data_dictionary.json'}")
print(f"  {OUT_DIR / 'dropped_columns_log.csv'}")
print(f"  {OUT_DIR / 'missing_code_recode_log.csv'}")


Saved:
  /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/child_recode_clean.csv  (19,530 rows x 38 columns)
  /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/data_dictionary.json
  /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/dropped_columns_log.csv
  /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/missing_code_recode_log.csv



## 13. A human-readable version, for exploration and reporting

`child_recode_clean.csv` keeps DHS's numeric codes (`v106`, `v113`, ...)
because that's what Part 3's feature engineering and Part 4's encoding
need to work with. But codes aren't how anyone wants to *read* the data —
counties as numbers, education as 0-3, wealth as 1-5. This step builds a
companion table with descriptive column names and every category decoded
to its actual text (e.g. `v113 == 11` -> `source of drinking water ==
"piped into dwelling"`), for use in Part 2's exploratory analysis and any
reporting or presentation.

Two continuous measures also get converted to their natural units:
birth weight from grams-implied storage to kilograms, and the HAZ score
from *100-scaled storage to actual standard deviations — plus the
`stunting category` label from the severity scheme above.

This table is for humans, not `scikit-learn` — Parts 3-4 continue to work
from the coded version.


In [22]:

HAZ_CATEGORY_BINS = [-6.01, -3, -2, 6.01]  # small buffer past the documented -6/6 endpoints so they're included
HAZ_CATEGORY_LABELS = ["Severe stunting", "Moderate stunting", "Normal growth"]

# variable -> descriptive column name
RENAME_MAP = {
    "child_id": "child id",
    "caseid": "case id",
    "v024": "region",
    "v025": "type of residence",
    "v106": "highest level of education",
    "v113": "source of drinking water",
    "v116": "type of toilet facility",
    "v151": "sex of household head",
    "v190": "wealth index",
    "b0": "child is twin",
    "b4": "sex of child",
    "bord": "birth order",
    "b11": "preceding birth interval (months)",
    "hw1": "child's age in months",
    "m18": "size of child at birth",
    "m4": "duration of breastfeeding",
    "m13": "month of pregnancy at first antenatal visit",
    "m14": "number of antenatal care visits",
    "m45": "took iron tablets during pregnancy",
    "h10": "had fever in last two weeks",
    "h11": "had diarrhea recently",
    "h33": "received vitamin a supplement",
}

# Variables in RENAME_MAP whose raw codes need decoding to text (the rest -
# birth order, age in months, preceding birth interval - are already plain numbers)
categorical_source_vars = [
    "v024", "v025", "v106", "v113", "v116", "v151", "v190",
    "b0", "b4", "m18", "m4", "m45", "h10", "h11", "h33",
]

labeled_df = df[list(RENAME_MAP.keys()) + ["m19", "hw70"]].copy()

for var in categorical_source_vars:
    labeled_df[var] = labeled_df[var].map(value_labels[var]).str.title()

# Natural units for the two continuous health measures
labeled_df["birth weight (kg)"] = labeled_df.pop("m19") / 1000
labeled_df["height-for-age z-score"] = labeled_df.pop("hw70") / 100
labeled_df["stunting category"] = pd.cut(
    labeled_df["height-for-age z-score"], bins=HAZ_CATEGORY_BINS, labels=HAZ_CATEGORY_LABELS
)

labeled_df = labeled_df.rename(columns=RENAME_MAP)

out_labeled_csv = OUT_DIR / "child_recode_labeled.csv"
labeled_df.to_csv(out_labeled_csv, index=False)

print(f"Saved: {out_labeled_csv}  ({labeled_df.shape[0]:,} rows x {labeled_df.shape[1]:,} columns)")
labeled_df.head(5)


Saved: /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/child_recode_labeled.csv  (19,530 rows x 25 columns)


,child id,case id,region,type of residence,highest level of education,source of drinking water,type of toilet facility,sex of household head,wealth index,child is twin,sex of child,birth order,preceding birth interval (months),child's age in months,size of child at birth,duration of breastfeeding,month of pregnancy at first antenatal visit,number of antenatal care visits,took iron tablets during pregnancy,had fever in last two weeks,had diarrhea recently,received vitamin a supplement,birth weight (kg),height-for-age z-score,stunting category
0,1 4 2_1,1 4 2,Mombasa,Urban,No Education,Public Tap/Standpipe,Pit Latrine With Slab,Male,Richer,Single Birth,Male,4,92,8,Larger Than Average,Still Breastfeeding,5,5,Yes,NaN,No,No,4.0,3.62,Normal growth
1,1 13 2_1,1 13 2,Mombasa,Urban,Secondary,Piped To Yard/Plot,Flush To Septic Tank,Male,Richest,Single Birth,Female,5,30,36,NaN,NaN,<NA>,<NA>,NaN,NaN,No,NaN,<NA>,-0.11,Normal growth
2,1 26 2_1,1 26 2,Mombasa,Urban,Secondary,Piped To Neighbor,Ventilated Improved Pit Latrine (Vip),Male,Richer,Single Birth,Male,3,23,58,NaN,NaN,<NA>,<NA>,NaN,NaN,No,NaN,<NA>,-0.86,Normal growth
3,1 42 1_1,1 42 1,Mombasa,Urban,Secondary,Public Tap/Standpipe,Ventilated Improved Pit Latrine (Vip),Female,Richest,Single Birth,Female,1,<NA>,45,NaN,NaN,<NA>,<NA>,NaN,NaN,No,NaN,<NA>,0.2,Normal growth
4,1 55 2_1,1 55 2,Mombasa,Urban,Secondary,Public Tap/Standpipe,Ventilated Improved Pit Latrine (Vip),Male,Richest,Single Birth,Male,2,23,4,NaN,Still Breastfeeding,6,3,Yes,NaN,No,No,<NA>,1.39,Normal growth



# Part 3: Feature Engineering

This part turns the cleaned raw variables into a model-ready feature table:
fixing a couple of placeholder codes the general-purpose cleaning in Part 1
was too conservative to touch, deriving clinically meaningful features
(BMI, low birth weight, birth spacing, breastfeeding status...), and
encoding categorical predictors. Every derived feature is documented with
the formula/rule used to build it.

Like Part 2, this reloads from disk so it can be run on its own.


In [23]:

df = pd.read_csv(OUT_DIR / "child_recode_clean.csv")
fe_log = []  # (feature, source_column(s), rule) — written out at the end for documentation

def log_feature(name, source, rule):
    fe_log.append({"feature": name, "source_columns": source, "rule": rule})



## 3.1 Close a gap the general-purpose cleaning missed

Part 1's missing-code recoding (step 6) matched codes whose value label
contained a phrase like "refused" or "not present" — a deliberately
conservative rule so it wouldn't guess. It looked at `v437`/`v438`
(mother's weight/height) and correctly cleared codes `9994` ("not present")
and `9995` ("refused"), but code `9996` ("other") doesn't match any of
those phrases and slipped through. For a body measurement, "other" is not
a numeric answer either, so we clear it here before using these columns to
build the maternal BMI feature.


In [24]:

for col in ["v437", "v438"]:
    n_other = (df[col] == 9996).sum()
    df.loc[df[col] == 9996, col] = np.nan
    print(f"{col}: cleared {n_other} residual 'other' placeholder codes")


v437: cleared 12 residual 'other' placeholder codes
v438: cleared 12 residual 'other' placeholder codes



## 3.2 Anthropometric / birth-history features


In [25]:

# --- Birth weight -----------------------------------------------------
# m19 is stored in kilograms * 1000 (3 implied decimals).
df["birthweight_kg"] = df["m19"] / 1000
df["low_birthweight"] = (df["birthweight_kg"] < 2.5).astype("Int8")
log_feature("birthweight_kg", "m19", "m19 / 1000 (stored *1000)")
log_feature("low_birthweight", "birthweight_kg", "1 if birthweight_kg < 2.5 kg (WHO low-birth-weight cutoff), else 0")

# --- Maternal BMI -------------------------------------------------------
# v437 (weight) and v438 (height) are stored *10 (1 implied decimal).
mother_weight_kg = df["v437"] / 10
mother_height_m = (df["v438"] / 10) / 100
df["mother_bmi"] = mother_weight_kg / (mother_height_m ** 2)
df["mother_thin"] = (df["mother_bmi"] < 18.5).astype("Int8")
df["mother_short_stature"] = (mother_height_m * 100 < 145).astype("Int8")
log_feature("mother_bmi", "v437, v438", "weight_kg / height_m^2, both /10 to undo DHS's *10 storage")
log_feature("mother_thin", "mother_bmi", "1 if BMI < 18.5 (WHO underweight cutoff), else 0")
log_feature("mother_short_stature", "v438", "1 if height < 145 cm (commonly used maternal short-stature cutoff), else 0")

df[["birthweight_kg", "low_birthweight", "mother_bmi", "mother_thin", "mother_short_stature"]].describe(include="all").T


,count,mean,std,min,25%,50%,75%,max
birthweight_kg,4755.0,3.18076,0.622111,0.6,2.9,3.2,3.5,6.0
low_birthweight,19530.0,0.021966,0.146577,0.0,0.0,0.0,0.0,1.0
mother_bmi,10173.0,23.605726,5.246463,13.372749,19.852641,22.578489,26.372401,63.038474
mother_thin,19530.0,0.072453,0.259243,0.0,0.0,0.0,0.0,1.0
mother_short_stature,19530.0,0.002407,0.048999,0.0,0.0,0.0,0.0,1.0



## 3.3 Birth order and spacing features


In [26]:

# b11 (preceding birth interval, months) is NaN for first-born children,
# which is structural (there is no preceding birth), not a data problem.
df["short_birth_interval"] = np.where(df["b11"].isna(), np.nan, (df["b11"] < 24).astype(float))
df["high_birth_order"] = (df["bord"] >= 4).astype("Int8")
df["is_twin_or_multiple"] = (df["b0"] > 0).astype("Int8")

log_feature("short_birth_interval", "b11", "1 if preceding birth interval < 24 months (WHO spacing guidance), NaN for first births")
log_feature("high_birth_order", "bord", "1 if this is the 4th or later live birth, else 0")
log_feature("is_twin_or_multiple", "b0", "1 if child is part of a multiple birth (b0 > 0), else 0")

df[["short_birth_interval", "high_birth_order", "is_twin_or_multiple"]].mean()


,0
short_birth_interval,0.209647
high_birth_order,0.350691
is_twin_or_multiple,0.031951



## 3.4 Antenatal care and supplementation features


In [27]:

df["anc_adequate"] = (df["m14"] >= 4).astype("Int8")           # WHO's older "4+ visits" benchmark
df["anc_early_start"] = (df["m13"] <= 3).astype("Int8")        # first antenatal visit in the 1st trimester
df["iron_supplemented"] = df["m45"].astype("Int8")             # already a clean 0/1
df["vitamin_a_received"] = df["h33"].isin([1, 2, 3]).astype("Int8")  # any documented/recalled receipt

log_feature("anc_adequate", "m14", "1 if 4+ antenatal visits reported, else 0")
log_feature("anc_early_start", "m13", "1 if first antenatal visit was in month 1-3, else 0")
log_feature("iron_supplemented", "m45", "carried over as-is (already 0/1)")
log_feature("vitamin_a_received", "h33", "1 if any Vitamin A receipt documented (card date, card mark, or mother-reported), else 0")

df[["anc_adequate", "anc_early_start", "iron_supplemented", "vitamin_a_received"]].mean()


,0
anc_adequate,0.332207
anc_early_start,0.142448
iron_supplemented,0.896704
vitamin_a_received,0.375832



## 3.5 Breastfeeding features

`m4`/`m5` mix real durations with DHS status codes (`93` ever-but-not-
currently, `94` never breastfed, `95` still breastfeeding). We split that
into a clean categorical status and a numeric duration that's only
populated when a real duration exists.


In [28]:

BF_STATUS_MAP = {93: "stopped", 94: "never", 95: "still_breastfeeding"}

df["breastfeeding_status"] = df["m4"].map(BF_STATUS_MAP)
df.loc[df["breastfeeding_status"].isna() & df["m4"].notna(), "breastfeeding_status"] = "stopped"  # any leftover numeric m4 duration

# m5 holds the numeric duration in months for resolvable cases; special
# codes 93/94 that survived Part 1's cleaning aren't real durations.
df["breastfeeding_duration_months"] = df["m5"].where(~df["m5"].isin([93, 94]))

log_feature("breastfeeding_status", "m4", "categorical: never / stopped / still_breastfeeding, decoded from DHS status codes 93-95")
log_feature("breastfeeding_duration_months", "m5", "m5 kept as-is except codes 93/94 (non-numeric statuses) set to NaN")

df[["breastfeeding_status", "breastfeeding_duration_months"]].describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
breastfeeding_status,11728,3,still_breastfeeding,6788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
breastfeeding_duration_months,6788.0,NaN,NaN,NaN,10.815999,7.487932,0.0,5.0,10.0,16.0,35.0



## 3.6 Birth size feature


In [29]:

# m18 is already a clean 1 (very large) .. 5 (very small) ordinal scale; add a simple binary flag on top.
df["small_at_birth"] = df["m18"].isin([4, 5]).astype("Int8")
log_feature("small_at_birth", "m18", "1 if size at birth reported as 'smaller than average' or 'very small', else 0")
df["small_at_birth"].mean()


np.float64(0.03922171018945213)


## 3.7 Water and sanitation (WASH) features

`v113` and `v116` use dozens of specific source/facility codes. We collapse
them to the standard JMP-style **improved / unimproved** classification
used in WHO/UNICEF water and sanitation reporting, which is far more
useful to a model than 15-30 sparse categories.

> Assumption: bottled water and "flush, don't know where" are counted as
> improved, following common JMP practice; results can be revisited if a
> stricter definition is preferred.


In [30]:

IMPROVED_WATER_CODES = {10, 11, 12, 13, 14, 20, 21, 31, 41, 51, 71}     # piped, borehole, protected well/spring, rainwater, bottled
IMPROVED_SANITATION_CODES = {11, 12, 13, 15, 21, 22, 41}                # flush variants, VIP, slab pit latrine, composting

df["improved_water_source"] = df["v113"].isin(IMPROVED_WATER_CODES).astype("Int8")
df.loc[df["v113"].isna(), "improved_water_source"] = np.nan

df["improved_sanitation"] = df["v116"].isin(IMPROVED_SANITATION_CODES).astype("Int8")
df.loc[df["v116"].isna(), "improved_sanitation"] = np.nan

df["is_urban"] = (df["v025"] == 1).astype("Int8")

log_feature("improved_water_source", "v113", "1 if water source is JMP-'improved' (piped/borehole/protected well or spring/rainwater/bottled), else 0")
log_feature("improved_sanitation", "v116", "1 if toilet facility is JMP-'improved' (flush variants/VIP/slab pit latrine/composting), else 0")
log_feature("is_urban", "v025", "1 if residence is urban, 0 if rural")

df[["improved_water_source", "improved_sanitation", "is_urban"]].mean()


,0
improved_water_source,0.660573
improved_sanitation,0.580027
is_urban,0.342345



## 3.8 Encode remaining categorical predictors

- **Ordinal** variables (education, wealth) are kept as their existing
  numeric codes — the order is already meaningful, so one-hot encoding
  would throw information away.
- **Binary** variables are recoded to a clean, self-explanatory 0/1.
- **Nominal** variables with more than two unlabeled categories (region)
  are one-hot encoded, since there's no natural ordering to preserve.


In [31]:

# Binary recodes -> clean, self-documenting 0/1
df["is_female"] = (df["b4"] == 2).astype("Int8")
df["female_headed_household"] = (df["v151"] == 2).astype("Int8")
# h11 (diarrhea) is already 0/1 in this file; included here for completeness
df["had_diarrhea_recently"] = df["h11"].astype("Int8")

log_feature("is_female", "b4", "1 if female, 0 if male")
log_feature("female_headed_household", "v151", "1 if household head is female, 0 if male")
log_feature("had_diarrhea_recently", "h11", "carried over as-is (already 0/1)")

# One-hot encode region (nominal, 47 counties) - prefix keeps columns identifiable
region_dummies = pd.get_dummies(df["v024"], prefix="region", dtype="Int8")
df = pd.concat([df, region_dummies], axis=1)
log_feature(f"region_* ({region_dummies.shape[1]} columns)", "v024", "one-hot encoded county of residence")

print(f"Added {region_dummies.shape[1]} one-hot region columns.")


Added 47 one-hot region columns.



## 3.9 HAZ severity category label

Alongside the continuous `hw70` z-score, we add the three-level severity
label introduced in Part 2's EDA (`Severe stunting` / `Moderate stunting`
/ `Normal growth`), so it's available directly from the saved table rather
than needing to be recomputed downstream.


In [32]:

HAZ_CATEGORY_BINS = [-6.01, -3, -2, 6.01]
HAZ_CATEGORY_LABELS = ["Severe stunting", "Moderate stunting", "Normal growth"]

df["haz_category"] = pd.cut(df["hw70"] / 100, bins=HAZ_CATEGORY_BINS, labels=HAZ_CATEGORY_LABELS)
log_feature("haz_category", "hw70",
            "Severe stunting: HAZ < -3 | Moderate stunting: -3 <= HAZ < -2 | Normal growth: HAZ >= -2")

df["haz_category"].value_counts()


,count
haz_category,
Normal growth,14176
Moderate stunting,2390
Severe stunting,761



## 3.10 Assemble the final model-ready table

Raw columns that have now been superseded by an engineered version are
dropped so the final table isn't carrying two copies of the same
information (e.g. `m19` -> `birthweight_kg`). Identifier columns and the
target are always kept.


In [33]:

target_col = "hw70"
id_cols = ["child_id", "caseid", "bidx"]

superseded_raw_cols = [
    "m19",            # -> birthweight_kg
    "v437", "v438",   # -> mother_bmi, mother_thin, mother_short_stature
    "m4", "m5",       # -> breastfeeding_status, breastfeeding_duration_months
    "b4", "v151", "h11",  # -> is_female, female_headed_household, had_diarrhea_recently
    "v024",           # -> region_* dummies
]
superseded_raw_cols = [c for c in superseded_raw_cols if c in df.columns]

# *_label helper columns were for human readability during EDA, not modeling inputs
label_cols = [c for c in df.columns if c.endswith("_label")]

drop_cols = superseded_raw_cols + label_cols
model_df = df.drop(columns=drop_cols)

print(f"Dropped {len(superseded_raw_cols)} raw columns superseded by engineered features: {superseded_raw_cols}")
print(f"Dropped {len(label_cols)} EDA-only *_label columns.")
print(f"Final model-ready table: {model_df.shape[0]:,} rows x {model_df.shape[1]:,} columns")
model_df.head(3)


Dropped 9 raw columns superseded by engineered features: ['m19', 'v437', 'v438', 'm4', 'm5', 'b4', 'v151', 'h11', 'v024']
Dropped 8 EDA-only *_label columns.
Final model-ready table: 19,530 rows x 90 columns


,child_id,caseid,bidx,hw70,hw1,bord,b0,b11,m18,m19a,v106,v133,m14,m13,m45,v190,v113,v116,v025,h33,h10,birthweight_kg,low_birthweight,mother_bmi,mother_thin,mother_short_stature,short_birth_interval,high_birth_order,is_twin_or_multiple,anc_adequate,...,region_19,region_20,region_21,region_22,region_23,region_24,region_25,region_26,region_27,region_28,region_29,region_30,region_31,region_32,region_33,region_34,region_35,region_36,region_37,region_38,region_39,region_40,region_41,region_42,region_43,region_44,region_45,region_46,region_47,haz_category
0,1 4 2_1,1 4 2,1,362.0,8.0,4,0,92.0,2.0,2.0,0,0,5.0,5.0,1.0,4,14,22.0,1,0.0,NaN,4.0,0,37.351305,0,0,0.0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Normal growth
1,1 13 2_1,1 13 2,1,-11.0,36.0,5,0,30.0,NaN,NaN,2,12,NaN,NaN,NaN,5,12,12.0,1,NaN,NaN,NaN,0,39.511578,0,0,0.0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Normal growth
2,1 26 2_1,1 26 2,1,-86.0,58.0,3,0,23.0,NaN,NaN,2,12,NaN,NaN,NaN,4,13,21.0,1,NaN,NaN,NaN,0,29.648743,0,0,1.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Normal growth



## 3.11 Save the feature-engineered dataset

Missing values in the engineered features are left as `NaN` rather than
imputed here — imputation strategy (median, model-based, etc.) is a
modeling-stage decision that depends on the algorithm being used, so it's
kept out of scope for this notebook. The feature engineering log records
exactly how each new column was derived.


In [40]:

model_csv = OUT_DIR / "child_recode_model_ready.csv"
model_df.to_csv(model_csv, index=False)

fe_log_df = pd.DataFrame(fe_log)
fe_log_df.to_csv(OUT_DIR / "feature_engineering_log.csv", index=False)

print("Saved:")
print(f"  {model_csv}  ({model_df.shape[0]:,} rows x {model_df.shape[1]:,} columns)")
print(f"  {OUT_DIR / 'feature_engineering_log.csv'}  ({len(fe_log_df)} engineered features documented)")
fe_log_df


Saved:
  /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/child_recode_model_ready.csv  (17,327 rows x 90 columns)
  /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/feature_engineering_log.csv  (23 engineered features documented)


,feature,source_columns,rule
0,birthweight_kg,m19,m19 / 1000 (stored *1000)
1,low_birthweight,birthweight_kg,1 if birthweight_kg < 2.5 kg (WHO low-birth-we...
2,mother_bmi,"v437, v438","weight_kg / height_m^2, both /10 to undo DHS's..."
3,mother_thin,mother_bmi,"1 if BMI < 18.5 (WHO underweight cutoff), else 0"
4,mother_short_stature,v438,1 if height < 145 cm (commonly used maternal s...
5,short_birth_interval,b11,1 if preceding birth interval < 24 months (WHO...
6,high_birth_order,bord,"1 if this is the 4th or later live birth, else 0"
7,is_twin_or_multiple,b0,1 if child is part of a multiple birth (b0 > 0...
8,anc_adequate,m14,"1 if 4+ antenatal visits reported, else 0"
9,anc_early_start,m13,"1 if first antenatal visit was in month 1-3, e..."



# Part 4: Feature Extraction and Representation

Parts 1-3 produced a clean, engineered table, but it isn't quite ready for
`scikit-learn` yet: the target is still a continuous z-score, one feature
(`breastfeeding_status`) is still text, and every feature still has
`NaN`s. This part turns `child_recode_model_ready.csv` into a numeric
matrix a classifier can consume.

**Framing the problem as classification.** The rest of this notebook
predicts the **HAZ severity category** introduced in Part 2/3
(`Severe stunting` / `Moderate stunting` / `Normal growth`) rather than the
continuous z-score, since "classification using supervised learning" and
the evaluation metrics that follow (precision/recall/ROC-AUC) are built
for a categorical target.


In [35]:

model_df = pd.read_csv(OUT_DIR / "child_recode_model_ready.csv")

# --- 4.1 Classification target ---------------------------------------
# haz_category was already derived in Part 3 (Severe stunting: HAZ < -3,
# Moderate stunting: -3 <= HAZ < -2, Normal growth: HAZ >= -2).

# A child with no valid anthropometry has no label at all - these rows
# can't be used for supervised learning and are dropped here (not earlier,
# since Parts 1-3 are about the data generally, not this specific task).
before_n = len(model_df)
model_df = model_df[model_df["haz_category"].notna()].reset_index(drop=True)
print(f"Dropped {before_n - len(model_df):,} rows with missing HAZ (no label). "
      f"{len(model_df):,} rows remain.")

print(model_df["haz_category"].value_counts(normalize=True).rename("share"))


Dropped 2,203 rows with missing HAZ (no label). 17,327 rows remain.
haz_category
Normal growth        0.818145
Moderate stunting    0.137935
Severe stunting      0.043920
Name: share, dtype: float64



## 4.2 Separate identifiers, target, and features

`hw70` (the raw z-score the label was derived from) is dropped from the
feature set — keeping it would let the model "cheat" by learning the exact
threshold rather than the underlying risk factors.


In [36]:

id_cols = ["child_id", "caseid", "bidx"]
target_col = "haz_category"
leakage_cols = ["hw70"]  # source of the label - must not be used as a feature

feature_cols = [c for c in model_df.columns if c not in id_cols + [target_col] + leakage_cols]

X = model_df[feature_cols].copy()
y = model_df[target_col].astype(str).copy()  # plain strings avoid categorical-dtype quirks in sklearn

categorical_features = ["breastfeeding_status"]
numeric_features = [c for c in feature_cols if c not in categorical_features]

print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]:,} columns "
      f"({len(numeric_features)} numeric, {len(categorical_features)} categorical)")


Feature matrix: 17,327 rows x 85 columns (84 numeric, 1 categorical)



## 4.3 Train/test split

Split before any imputation or scaling is fitted, so nothing about the
test set leaks into how those transformations are learned. Stratified on
the target since the three categories are quite imbalanced (roughly
73% / 12% / 4% normal/moderate/severe of valid measurements) — a plain
random split could otherwise shift that balance between splits.


In [37]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train class shares:")
print(y_train.value_counts(normalize=True))
print("\nTest class shares:")
print(y_test.value_counts(normalize=True))


Train class shares:
haz_category
Normal growth        0.818123
Moderate stunting    0.137941
Severe stunting      0.043936
Name: proportion, dtype: float64

Test class shares:
haz_category
Normal growth        0.818234
Moderate stunting    0.137911
Severe stunting      0.043855
Name: proportion, dtype: float64



## 4.4 Build the preprocessing pipeline

A single `ColumnTransformer`, fit only on the training data, handles the
remaining representation work:

- **Numeric features:** median imputation (robust to the outliers already
  seen in Part 2), then standardization — required for logistic regression,
  harmless for the tree-based models used later.
- **Categorical feature (`breastfeeding_status`):** missing values become
  their own explicit `"missing"` category (a child's mother not answering
  the breastfeeding question is itself informative), then one-hot encoding.

Wrapping this in a `Pipeline`/`ColumnTransformer` means every model below
gets an identical, leakage-safe representation, and the same object can
transform new data at prediction time.


In [38]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# Fit on train, inspect the resulting representation size
X_train_transformed = preprocessor.fit_transform(X_train)
print(f"Encoded feature matrix: {X_train_transformed.shape[1]} columns "
      f"(from {X.shape[1]} original features, after one-hot expansion)")


Encoded feature matrix: 88 columns (from 85 original features, after one-hot expansion)


In [39]:

# --- Bridge to 03_modelling.ipynb -----------------------------------------
# This notebook runs in its own kernel, so the objects built above (train/test
# split, fitted preprocessor, engineered table) are saved to disk here and
# reloaded at the top of 03_modelling.ipynb, rather than relying on shared
# in-memory state.
import joblib

bridge = {
    "X": X, "y": y,
    "X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
    "feature_cols": feature_cols,
    "numeric_features": numeric_features, "categorical_features": categorical_features,
    "preprocessor": preprocessor,
    "model_df": model_df,
    "HAZ_CATEGORY_LABELS": HAZ_CATEGORY_LABELS,
    "HAZ_CATEGORY_BINS": HAZ_CATEGORY_BINS,
}
bridge_path = OUT_DIR / "bridge_02_to_03.joblib"
joblib.dump(bridge, bridge_path)
print(f"Saved handoff objects for 03_modelling.ipynb -> {bridge_path}")


Saved handoff objects for 03_modelling.ipynb -> /content/drive/MyDrive/Ngao_labs/capstone/child recode/processed/bridge_02_to_03.joblib
